# Ch 3. 노트북에서 가능한 것

**노트북·Colab T4·사내 GPU 한 장의 메모리·연산·시간 예산을 모델 크기·토큰 수로 환산한다.**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part1/ch03_laptop_budget.ipynb)

In [ ]:
# !pip install -q torch

import torch

# Device 및 메모리 확인
if torch.cuda.is_available():
    device = "cuda"
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {vram:.1f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    print("Apple Silicon MPS available")
else:
    device = "cpu"
    print("CPU only")

print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 최소 예제 — 메모리 30초 산수

학습 시 메모리 식 (대략):

$$\text{Memory} \approx N \times (\text{params} + \text{grads} + \text{Adam}_1 + \text{Adam}_2 + \text{activation})$$

bf16 + Adam + grad로 보통 약 **14~16 byte/param** + activation.

- **10M 모델** → 약 1GB — M2(16GB), T4(16GB) 모두 넉넉
- **125M 모델** → 약 3~5GB — T4 가능
- **1B 모델** → 약 20GB+ — T4 학습 불가, A100부터

In [ ]:
# memory_budget.py
def training_memory_gb(N_params, dtype="bf16"):
    """대략 학습 메모리 추정 (activation 제외)."""
    bytes_per_param = 2 if dtype in ("bf16", "fp16") else 4
    # bf16 mixed: params + grads = 2+2, Adam = 4+4 = 12, total ≈ 14
    # fp32 pure : 4+4+4+4 = 16
    factor = 14 if dtype in ("bf16", "fp16") else 16
    return N_params * factor / 1e9  # GB

print(f"{'모델':10s}  {'bf16':>8s}  {'fp32':>8s}")
print("-" * 32)
for N, name in [(1e7, "10M"), (1.25e8, "125M"), (1e9, "1B"), (7e9, "7B")]:
    bf16 = training_memory_gb(N, 'bf16')
    fp32 = training_memory_gb(N, 'fp32')
    print(f"  {name:5s}  bf16: {bf16:6.2f} GB  fp32: {fp32:6.2f} GB")

In [ ]:
# activation 메모리 추정 (batch x seq x hidden x 12 대략)
def activation_memory_gb(batch_size, seq_len, hidden_dim, n_layers):
    """대략 activation 메모리 추정 (bf16 기준)."""
    # 레이어당 약 12 x hidden 텐서
    bytes_per_elem = 2  # bf16
    total = batch_size * seq_len * hidden_dim * n_layers * 12 * bytes_per_elem
    return total / 1e9

# 본 책 기준선: 10M 모델 (hidden=256, 6 layers)
act = activation_memory_gb(batch_size=16, seq_len=512, hidden_dim=256, n_layers=6)
print(f"10M 모델 activation (batch=16, seq=512): {act:.2f} GB")

# 합계
param_mem = training_memory_gb(1e7, 'bf16')
total = param_mem + act
print(f"합계 (params+grads+Adam+activation): ~{total:.2f} GB")

## 실전 — 본 책 기준선 산수

### 시간 = (총 FLOPs) / (장비 처리 FLOPs)

학습 1 토큰당 forward+backward FLOPs는 대략:

$$\text{FLOPs/token} \approx 6N$$

총 FLOPs:

$$\text{Total} \approx 6 \times N \times D$$

본 책 기준 (10M params, 200M tokens):

$$6 \times 10^7 \times 2 \times 10^8 = 1.2 \times 10^{16} \text{ FLOPs}$$

**기준선**: M2 Pro MPS 또는 Colab T4면 **수십 분 ~ 1시간**. 책 제목의 "4시간"은 toolchain 셋업·평가·디버깅 포함 보수적 추정.

In [ ]:
# time_budget.py
def hours_to_train(N_params, D_tokens, effective_tflops):
    flops = 6 * N_params * D_tokens
    seconds = flops / (effective_tflops * 1e12)
    return seconds / 3600

scenarios = [
    ("10M  · 200M  · M2 Pro MPS", 1e7,    2e8,   3),
    ("10M  · 200M  · T4",          1e7,    2e8,   20),
    ("30M  · 600M  · T4",          3e7,    6e8,   20),
    ("125M · 2.5B  · T4",          1.25e8, 2.5e9, 20),
    ("125M · 2.5B  · A100",        1.25e8, 2.5e9, 150),
]
print(f"{'시나리오':35s}  {'시간':>8s}")
print("-" * 48)
for name, N, D, tf in scenarios:
    h = hours_to_train(N, D, tf)
    print(f"  {name:35s}  {h:6.2f} h")

In [ ]:
# Chinchilla 비율 시각화
# D ≈ 20 × N (compute-optimal)

models = {
    "Chinchilla 70B": (70e9, 1.4e12),
    "Llama 3 8B":     (8e9,  15e12),
    "SmolLM2 1.7B":  (1.7e9, 11e12),
    "본 책 10M":      (1e7,  2e8),
}

print(f"{'모델':15s}  {'파라미터':>12s}  {'학습 토큰':>12s}  {'비율 (D/N)':>12s}")
print("-" * 58)
for name, (N, D) in models.items():
    ratio = D / N
    print(f"  {name:15s}  {N/1e9:10.1f}B  {D/1e9:10.1f}B  {ratio:12.0f}x")

## 연습문제

1. 본인 노트북의 RAM·CPU·GPU 정보를 적고, `training_memory_gb` 함수로 학습 가능한 최대 모델 크기를 추정하라. activation 마진 30% 포함.
2. **30M 모델 + Chinchilla 비율 600M 토큰**을 본인 환경에서 돌리면 몇 시간 걸리나? `hours_to_train`으로 계산. 12시간 안에 들어오나?
3. spec FLOPS와 실효 FLOPS 차이가 30~50%라고 했다. 본인 장비로 작은 학습 (10M 모델, 1000 step)을 돌려 **실측 토큰/초**를 측정하고, spec 대비 몇 %인지 계산.
4. **(생각해볼 것)** Chinchilla 비율을 무시하고 일부러 over-train (10M · 1B 토큰, 100×) 한다면 메모리·시간이 어떻게 바뀌는가? 어느 쪽이 한계가 되겠는가?

In [ ]:
# 연습문제 1 — 본인 장비 메모리 기준 최대 모델 크기 추정
import torch

# 본인 VRAM 또는 RAM 입력
if torch.cuda.is_available():
    available_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU VRAM: {available_gb:.1f} GB")
else:
    available_gb = 8  # 본인 RAM 직접 입력 (GB)
    print(f"가정한 RAM: {available_gb} GB (직접 수정하세요)")

# 30% 마진 후 사용 가능 메모리
usable_gb = available_gb * 0.7
# activation 추정 제외 (params+grads+Adam 만)
max_params = usable_gb * 1e9 / 14  # 14 bytes/param (bf16)
print(f"\n사용 가능 메모리 (30% 마진): {usable_gb:.1f} GB")
print(f"최대 파라미터 수 (activation 제외): {max_params/1e6:.0f}M")
print("(activation 포함 시 실제 최대치는 더 작음)")

In [ ]:
# 연습문제 2 — 30M + 600M 토큰 시간 계산
# 본인 장비의 실효 TFLOPS 입력
# M2 Pro MPS ≈ 3, T4 ≈ 20, A100 ≈ 150
my_tflops = 3  # 본인 장비에 맞게 수정

h = hours_to_train(N_params=3e7, D_tokens=6e8, effective_tflops=my_tflops)
print(f"30M 모델, 600M 토큰, {my_tflops} TFLOPS: {h:.2f} h")
print(f"Colab 무료 12시간 안? {'YES' if h < 12 else 'NO'}")

In [ ]:
# 연습문제 3 — 실측 토큰/초 측정 (간단한 forward 속도)
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "HuggingFaceTB/SmolLM2-135M"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16)

# 간단한 forward 속도 측정 (추론 기준)
batch_size = 4
seq_len = 128
n_steps = 50

dummy_ids = torch.randint(0, 1000, (batch_size, seq_len))

# warmup
with torch.no_grad():
    for _ in range(5):
        _ = model(dummy_ids).logits

start = time.time()
with torch.no_grad():
    for _ in range(n_steps):
        _ = model(dummy_ids).logits
elapsed = time.time() - start

tokens_per_sec = batch_size * seq_len * n_steps / elapsed
print(f"추론 속도: {tokens_per_sec:,.0f} tokens/sec")
print(f"(학습은 backward 포함으로 약 3~5배 느림)")

In [ ]:
# 연습문제 4 — over-train 시나리오 분석
# 10M · 1B 토큰 (100×)
print("=== over-train 시나리오 (10M · 1B 토큰) ===")
for label, tflops in [("M2 Pro MPS", 3), ("T4", 20), ("A100", 150)]:
    h = hours_to_train(1e7, 1e9, tflops)
    print(f"  {label}: {h:.2f} h")

print("\n메모리는? (10M 모델은 1GB 수준으로 변화 없음)")
print(training_memory_gb(1e7, 'bf16'), "GB (params+grads+Adam 기준)")
print("\n결론: 메모리는 괜찮지만 시간이 bottleneck")